# DNA Methylation Gene Heatmap
Visualize logFC / nlogp of DNA-methylation related genes across cell types using `ComplexHeatmap` (`PyComplexHeatmap`).

- Rows: genes grouped by Writers / Co-factors / Readers / Erasers / BER_Repair
- Columns: cell types (Region_Subclass)
- Values: logFC (or signed -log10(FDR))

In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import PyComplexHeatmap as pch
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

## DNA methylation gene dictionary

In [ ]:
dna_methylation_dict = {
    "Writers":     ["Dnmt1", "Dnmt3a", "Dnmt3b"],
    "Co-factors":  ["Dnmt3l", "Uhrf1", "Uhrf2"],
    "Readers":     ["Mecp2", "Mbd1", "Mbd2", "Mbd3", "Mbd4"],
    "Erasers":     ["Tet1", "Tet2", "Tet3"],
    "BER_Repair":  ["Tdg", "Neil1", "Neil2", "Smug1", "Mutyh", "Mbd4"],
}

# Build a long-form table with category labels
dna_methylation_long = (
    pd.DataFrame(
        [(category, gene) for category, genes in dna_methylation_dict.items() for gene in genes],
        columns=["Category", "Gene"],
    )
)
dna_methylation_long = dna_methylation_long.drop_duplicates(subset=["Gene"]).reset_index(drop=True)
dna_methylation_long

## Load DEG table

In [ ]:
DEG_XLSX = "/data2st2/junyi/output/1-1 six_dataset_DEG_list_fdr_FC01_filtered_7region_rmMB 20251227.xlsx"
df_deg_all = pd.read_excel(DEG_XLSX, sheet_name="all", index_col=0)
df_deg_all.head()

In [ ]:
print("Shape:", df_deg_all.shape)
print("Columns:", df_deg_all.columns.tolist())
print("Models:", df_deg_all["Model"].unique() if "Model" in df_deg_all.columns else "N/A")
print("Regions:", df_deg_all["Region"].unique() if "Region" in df_deg_all.columns else "N/A")

In [ ]:
# Make gene symbols case-insensitive
df_deg_all = df_deg_all.copy()
df_deg_all["Gene_upper"] = df_deg_all.index.astype(str).str.upper()
dna_methylation_long["Gene_upper"] = dna_methylation_long["Gene"].astype(str).str.upper()

## Filter to DNA methylation genes

In [ ]:
df_meth = df_deg_all[df_deg_all["Gene_upper"].isin(dna_methylation_long["Gene_upper"])].copy()
df_meth = df_meth.merge(
    dna_methylation_long[["Category", "Gene_upper"]],
    left_on="Gene_upper",
    right_on="Gene_upper",
    how="left",
)
df_meth["Gene"] = df_meth["Gene_upper"]
print("Filtered shape:", df_meth.shape)
df_meth.head()

## Build cell-type column

In [ ]:
# Construct a cell-type identifier (Region_Subclass) and clean it
df_meth["ctname"] = (
    df_meth["Region"].astype(str)
    + "_"
    + df_meth["Subclass"].astype(str).str.replace("/", "-", regex=False).str.replace(" ", "_", regex=False)
)
df_meth["ctname"] = df_meth["ctname"].str.replace("HIP", "HPF", regex=False)
df_meth = df_meth.dropna(subset=["ctname", "Gene", "log2FC", "FDR"])
df_meth.head()

## Compute nlogp (signed -log10(FDR))

In [ ]:
df_meth["nlogp"] = -1.0 * np.log10(df_meth["FDR"].astype(float) + 1e-300)
df_meth.loc[df_meth["Direction"].astype(str).str.lower() == "down", "nlogp"] *= -1
df_meth["log2FC"] = df_meth["log2FC"].astype(float)

## Build the heatmap matrix (genes × cell types)

In [ ]:
# Aggregate duplicates: take the max absolute effect per gene/ctname
def _agg(values):
    arr = values.dropna()
    if arr.empty:
        return np.nan
    # Use the value with the largest absolute magnitude (preserves up/down)
    return arr.loc[arr.abs().idxmax()]

mat_logfc = (
    df_meth
    .pivot_table(index="Gene", columns="ctname", values="log2FC", aggfunc=_agg)
)
mat_nlogp = (
    df_meth
    .pivot_table(index="Gene", columns="ctname", values="nlogp", aggfunc=_agg)
)

# Sort genes by category
gene_order = (
    dna_methylation_long
    .set_index("Gene_upper")
    .loc[mat_logfc.index]
    .reset_index()
    .rename(columns={"Gene_upper": "Gene"})
)
gene_order["Category"] = pd.Categorical(
    gene_order["Category"],
    categories=list(dna_methylation_dict.keys()),
    ordered=True,
)
gene_order = gene_order.sort_values(["Category", "Gene"]).reset_index(drop=True)
mat_logfc = mat_logfc.loc[gene_order["Gene"].values]
mat_nlogp = mat_nlogp.loc[gene_order["Gene"].values]
mat_logfc

## Build column metadata (Region / Neurotransmitter / Sex)

In [ ]:
# Read the meta file to enrich columns with Region / Neurotransmitter / Sex (if available)
META_CSV = "/data2st2/junyi/output/stg1028/combined_ALL_meta.csv"
if os.path.exists(META_CSV):
    df_meta = pd.read_csv(META_CSV, index_col=0)
    df_meta["ctname"] = (
        df_meta["region"].astype(str)
        + "_"
        + df_meta["celltype.L2"].astype(str).str.replace("/", "-", regex=False).str.replace(" ", "_", regex=False)
    )
    df_meta["ctname"] = df_meta["ctname"].str.replace("HIP", "HPF", regex=False)
    df_meta["Neurotransmitter"] = df_meta["Neurotransmitter_celltype"].fillna("NN")
    col_meta = (
        df_meta[["ctname", "region", "Neurotransmitter", "sex"]]
        .dropna(subset=["ctname"])
        .drop_duplicates(subset=["ctname"])
        .set_index("ctname")
    )
else:
    col_meta = pd.DataFrame(index=mat_logfc.columns)
    col_meta["region"] = [c.split("_")[0] for c in mat_logfc.columns]
    col_meta["Neurotransmitter"] = "NN"
    col_meta["sex"] = "M"

# Align with matrix columns
col_meta = col_meta.reindex(mat_logfc.columns)

# Sort columns by region / sex / neurotransmitter
region_order = ["AMY", "HPF", "HY", "iCTX", "PFC", "STR", "TH"]
col_meta = col_meta.copy()
col_meta["region"] = pd.Categorical(col_meta["region"], categories=region_order, ordered=True)
col_meta = col_meta.sort_values(["region", "sex", "Neurotransmitter"])
mat_logfc = mat_logfc[col_meta.index]
mat_nlogp = mat_nlogp[col_meta.index]
col_meta

## Color schemes

In [ ]:
region_colors = {
    "AMY": "#6DA1D5",
    "HPF": "#E13127",
    "HY":  "#D88A91",
    "iCTX": "#82C341",
    "PFC": "#F57E20",
    "STR": "#8A60AA",
    "TH":  "#863220",
}

nt_colors = {
    "Glutamatergic": "#FFC000",
    "GABAergic":     "#00B050",
    "Dopaminergic":  "#ff7f0e",
    "Cholinergic":   "#1f77b4",
    "Serotonergic":  "#e377c2",
    "Histaminergic": "#aa40fc",
    "NN":            "#8c564b",
}

sex_colors = {"M": "#0080FF", "F": "#E800E8"}

category_colors = {
    "Writers":    "#E41A1C",
    "Co-factors": "#377EB8",
    "Readers":    "#4DAF4A",
    "Erasers":    "#984EA3",
    "BER_Repair": "#FF7F00",
}

## Plot log2FC heatmap

In [ ]:
OUTPUT_DIR = "/data2st2/junyi/output/stg1028/dna_methylation"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def plot_methylation_heatmap(mat, col_meta, gene_order, value_name="log2FC",
                              pdf_path=None, clip=None, figsize=None):
    plot_mat = mat.copy()
    if clip is not None:
        plot_mat = plot_mat.clip(clip[0], clip[1])

    row_meta = gene_order.set_index("Gene")[["Category"]]

    region_series = col_meta["region"].astype(str)
    nt_series = col_meta["Neurotransmitter"].astype(str) if "Neurotransmitter" in col_meta.columns else None
    sex_series = col_meta["sex"].astype(str) if "sex" in col_meta.columns else None

    annotations = [pch.anno_simple(region_series, colors=region_colors, add_text=True, axis=1)]
    if nt_series is not None:
        annotations.append(pch.anno_simple(nt_series, colors=nt_colors, add_text=False, axis=1))
    if sex_series is not None:
        annotations.append(pch.anno_simple(sex_series, colors=sex_colors, add_text=False, axis=1))

    col_ha = pch.HeatmapAnnotation(
        Region=annotations[0],
        Neurotransmitter=annotations[1] if len(annotations) > 1 else None,
        Sex=annotations[2] if len(annotations) > 2 else None,
        axis=1,
    )

    row_ha = pch.HeatmapAnnotation(
        Category=pch.anno_simple(
            row_meta["Category"],
            colors=category_colors,
            add_text=True,
            axis=0,
        ),
        axis=0,
    )

    if figsize is None:
        figsize = (max(10, plot_mat.shape[1] * 0.45 + 4), max(4, plot_mat.shape[0] * 0.45 + 4))

    plt.figure(figsize=figsize)

    cmap = LinearSegmentedColormap.from_list("blue_red", ["blue", "white", "red"])

    cm = pch.ClusterMapPlotter(
        data=plot_mat,
        top_annotation=col_ha,
        left_annotation=row_ha,
        row_cluster=False,
        col_cluster=False,
        row_dendrogram=False,
        col_dendrogram=False,
        show_rownames=True,
        show_colnames=True,
        cmap=cmap,
        center=0,
        label=value_name,
        yticklabels_kws={"labelsize": 9},
        xticklabels_kws={"labelsize": 6, "rotation": 90},
        rasterized=True,
    )

    if hasattr(cm, "ax_row_names"):
        for text in cm.ax_row_names.texts:
            text.set_fontsize(9)
            text.set_rotation(0)
            text.set_clip_on(False)
    if hasattr(cm, "ax_col_names"):
        for text in cm.ax_col_names.texts:
            text.set_fontsize(6)
            text.set_rotation(90)
            text.set_clip_on(False)

    plt.subplots_adjust(left=0.18, bottom=0.18, right=0.95, top=0.95)
    if pdf_path is not None:
        plt.savefig(pdf_path, bbox_inches="tight", pad_inches=0.3, dpi=300)
    plt.show()
    plt.close()

In [ ]:
plot_methylation_heatmap(
    mat=mat_logfc,
    col_meta=col_meta,
    gene_order=gene_order,
    value_name="log2FC",
    pdf_path=os.path.join(OUTPUT_DIR, "dna_methylation_log2FC_heatmap.pdf"),
    clip=(-1.5, 1.5),
)

## Plot signed -log10(FDR) heatmap

In [ ]:
plot_methylation_heatmap(
    mat=mat_nlogp,
    col_meta=col_meta,
    gene_order=gene_order,
    value_name="signed -log10(FDR)",
    pdf_path=os.path.join(OUTPUT_DIR, "dna_methylation_nlogp_heatmap.pdf"),
    clip=(-10, 10),
)

## Save the prepared tables

In [ ]:
mat_logfc.to_csv(os.path.join(OUTPUT_DIR, "dna_methylation_log2FC_matrix.csv"))
mat_nlogp.to_csv(os.path.join(OUTPUT_DIR, "dna_methylation_nlogp_matrix.csv"))
df_meth.to_csv(os.path.join(OUTPUT_DIR, "dna_methylation_long.csv"), index=False)
print(f"Saved outputs to {OUTPUT_DIR}")